Updated from https://github.com/DLR-MF-DAS/embed2scale-challenge-supplement/blob/main/data_loading_submission_demo/baseline_compression_mean.ipynb

### NeuCo-Bench Mean Baseline

This notebook creates baseline embeddings by bilinear interpolation and averaging of the modalities.

We use the SSL4EODownstreamDataset to load the data. The datacubes are of shape (1, 4, 27, 264, 264), (number of samples, number of timesteps, number of channels, height, width).

The embedding works as follow:
1. Subsample each channel to 8x8 pixels using bilinear interpolation -> shape (1, 4, 27, 8, 8)
2. Average B01 through B09 for both S2L1C and L2 L2A along the channel dimension. Average B11 and B12 along the channel dimension. Average S1 channels along the channel dimension. Concatenate the three averages and B10 along channel dimension -> shape (1, 4, 4, 8, 8)
3. Flatten into 1024 element vector -> shape (1024,)

After embedding, a submission file is created in the expected format for NeuCo-Bench. At the end, a function tests that a submission file has the right format.

In [2]:
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from scipy.ndimage import zoom
from torchvision import transforms

from data.dataset import SSL4EODownstreamDataset, S2L1C_MEAN, S2L1C_STD, S2L2A_MEAN, S2L2A_STD, S1GRD_MEAN, S1GRD_STD

# Configurations

In [3]:
# Order of modalities.
# In this demo, modalities are ordered the same as the default order in the SSL4EOS12 dataset class.
# Modalities are loaded in the order provided here.
modalities = ['s2l1c', 's2l2a', 's1grd']

# Path to challenge data folder, i.e. the folder containing the s1, s2l1c and s2l2a subfolders.
path_to_data = 'path/to/data/'

# Path to where the submission file should be written.
path_to_output_file = 'embeddings.csv'

write_result_to_file = True  # Set to True to trigger saving of the csv at the end.

# Get mean and standard deviations for the modalities
# Note that we will use the `shift_s2_channels` flag in the dataset, and 
# therefore use the mean and standard deviation of the SSL4EO-S12 v1.1 dataset.
mean_data = S2L1C_MEAN + S2L2A_MEAN + S1GRD_MEAN
std_data = S2L1C_STD + S2L2A_STD + S1GRD_STD

data_transform = transforms.Compose([
    # Add additional transformation here
    transforms.Normalize(mean=mean_data, std=std_data)
])

# Load data

In [4]:
# Concatenate modalities
# dataloader output is {'data': concatenated_data, 'file_name': file_name}
# The data has shapes [n_samples, n_seasons, n_channels, height, width] (for concatenated_data [1, 4, 27, 264, 264])

dataset = SSL4EODownstreamDataset(path_to_data,
                                  modalities = modalities, 
                                  dataset_name='bands', 
                                  transform=data_transform, 
                                  concat=False,
                                  output_file_name=True,
                                  shift_s2_channels=True
                                 )

# Print dataset length
print(f"Length of train dataset: {len(dataset)}")

# Print shape of first sample
for m, d in dataset[0]['data'].items():
    print(f'Modality {m} shape:', d.shape)

Length of train dataset: 13260
Modality s2l1c shape: torch.Size([1, 4, 13, 264, 264])
Modality s2l2a shape: torch.Size([1, 4, 12, 264, 264])
Modality s1grd shape: torch.Size([1, 4, 2, 264, 264])


# Create submission file

In [5]:
def create_submission_from_dict(emb_dict):
    """Assume dictionary has format {hash-id0: embedding0, hash-id1: embedding1, ...}
    """
    df_submission = pd.DataFrame.from_dict(emb_dict, orient='index')
    
    # Reset index with name 'id'
    df_submission.index.name = 'id'
    df_submission.reset_index(drop=False, inplace=True)
        
    return df_submission
        

# Compress by bilinear transform and channel averaging

In this section, we create a submission file by processing each sample accordingly:
1. Subsampling each channel to 8x8 pixels using bilinear interpolation
2. Average channels B01 to B09 for both L1C and L2A, average B11 and B12, and average S1 channels. Together with B10, this turns into 4 new channels.
3. Flatten into 1024 element vector.

In [6]:
# Correlation analysis show that L1C and L2A channels B01 to B09 are correlated, B11 and B12 are correlated, 
# and S1 VV and VH are correlated, so we average these, leaving (together with B10) 4 averaged channels.

def embed(data, file_name, emb_len=1024):
    # Bilinear interpolation of each channel separately.
    rescaled_mod = {m: zoom(d, (1, 1, 1, 8/d.shape[3], 8/d.shape[4]), order=1) for m, d in data.items()}

    # Calculate mean of correlated channels.
    b1_b9 = np.mean(np.concatenate((rescaled_mod['s2l1c'][:, :, 0:9, :, :], 
                                   rescaled_mod['s2l2a'][:, :, 0:9, :, :]), axis=2), 
                    axis=2, keepdims=True)
    b10 = rescaled_mod['s2l1c'][:, :, 9:10, :, :]
    b11_b12 = np.mean(np.concatenate((rescaled_mod['s2l1c'][:, :, 10:, :, :], 
                                     rescaled_mod['s2l2a'][:, :, 10:, :, :]), axis=2), 
                      axis=2, keepdims=True)
    s1 = np.mean(rescaled_mod['s1grd'], axis=2, keepdims=True)

    # Concatenate aggregated channels
    emb = np.concatenate((b1_b9, b10, b11_b12, s1), axis=2)

    # Flatten
    emb = emb.flatten()

    return {'file_name': file_name, 'embedding': emb}


def mean_embedding_parallel(dataset, n_workers=4, n_samples=None):
    
    # Initialize result embeddings
    embeddings = {}

    # Run embedding in parallel
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = []
        
        for ind, data_file_name in enumerate(dataset):
            data = data_file_name['data']
            # print(data)
            file_name = data_file_name['file_name']
            # Submit the batch for processing
            future = executor.submit(embed, data, file_name)
            futures.append(future)

            if (n_samples is not None) and (ind-1 > n_samples):
                break
        
        # Extract results
        for future in futures:
            res = future.result()
            # Compile embeddings
            embeddings[res['file_name']] = res['embedding']
    return embeddings


In [ ]:
n_workers = 1
if n_workers != 1:
    # Embed data
    embeddings = mean_embedding_parallel(dataset, n_workers=n_workers, n_samples=10)
else:
    embeddings = {}
    for ind, data_file_name in enumerate(dataset):
        data = data_file_name['data']
        file_name = data_file_name['file_name']
        emb = embed(data, file_name, 1024)
        embeddings[file_name] = emb['embedding']
        
        if ind % 100 == 0:
            print(f"Processed {ind} samples...")

In [8]:
# Create submission file
submission_file = create_submission_from_dict(embeddings)

In [ ]:
print('Number of embeddings:', len(submission_file))

In [10]:
submission_file.head()

,id,0,1,2,3,4,5,6,7,8,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,e5e970ccec58db07ef81d2d9b0ae6d65e0613278458769...,-0.173722,-0.066179,-0.348041,0.010026,-0.418046,-0.520044,-0.203427,-0.410088,-0.317312,...,0.260786,1.148047,-0.557547,-0.039418,0.440790,1.093749,-0.045903,0.812454,0.889676,0.863566
1,2eee5253b92f7c123922bab08111fa49eb187fd3617eb7...,0.313977,0.147601,0.145719,0.144143,0.138095,0.130194,0.087534,0.220588,-0.090096,...,-0.897500,-0.897307,-1.067629,-0.792969,-0.908441,-1.186697,-0.915565,-1.277131,-0.800818,-0.639083
2,1fd7df354c3c4a1870ac72dfcad752ecc7c14ebc9cac98...,6.479727,4.442508,-0.442519,-0.486496,-0.720285,-0.447445,-0.527314,-0.157729,3.918642,...,0.544118,0.609433,0.496540,1.149173,1.016606,1.329611,0.986724,1.614048,-1.605783,0.638078
3,dc0b9862201c4845062efcfaa6e7eff2e796791bfaf516...,-0.117202,-0.021037,-0.101366,0.122383,0.168067,-0.155037,0.196033,-0.136996,0.204254,...,1.874202,0.567171,0.668140,0.734213,0.856739,0.617233,-0.452738,0.256555,0.290383,1.193223
4,eb90badde02a80fae576ca2ee513c68991e1a0a36926ed...,-0.534713,-0.108874,0.142441,-0.259784,-0.136068,-0.240972,0.191771,0.095704,-0.219201,...,1.241990,0.943812,0.384504,0.452040,0.419828,0.727491,1.416173,0.405882,0.349547,1.225490


In [11]:
# Write submission
if write_result_to_file:
    submission_file.to_csv(path_to_output_file, index=False)

# Verify file integrity

In [15]:
from data.submission_utils import test_submission

embedding_ids = set(embeddings.keys())
embedding_dim = 1024

# Test submission
assert test_submission(path_to_output_file, embedding_ids, embedding_dim)